# 04 — Downstream signal evaluation

The question the project exists to answer: **does knowing the detector saw a pattern help
predict the next day's direction?**

Requires trained weights on the Hugging Face model repo (notebook 03) and the rendered
signal set (`python -m src.labeling.build_dataset`).

Whatever this shows is what gets published. A null result is the honest and more
interesting outcome, and the report generator words itself accordingly.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # import src/ from notebooks/
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import config
from src.data_pipeline.fetch_ohlc import fetch_ohlc
from src.downstream_signal.features import build_features
from src.downstream_signal.labels import next_day_direction, next_day_return
from src.downstream_signal.run_study import build_pattern_features
from src.downstream_signal.model import assemble_matrix

bars = fetch_ohlc(config.TICKER)
baseline = build_features(bars)
target = next_day_direction(bars)
fwd = next_day_return(bars)
pattern = build_pattern_features()   # cached after the first (slow) run
X, y = assemble_matrix(baseline, pattern, target)
print(f'{len(X)} rows x {X.shape[1]} features, {X.index.min().date()} -> {X.index.max().date()}')

## How often does the detector actually fire?

Check this before interpreting any null result. If the pattern columns were almost always
zero, "patterns do not help" would just mean "nothing was detected" — a different and much
less interesting finding.

In [ ]:
from src.detection.infer import PATTERN_FEATURES

share = (pattern.loc[X.index, 'det_any'] > 0).mean()
print(f'days with a detection on the latest candle: {share:.1%}')
print(f'mean detections per chart: {pattern.loc[X.index, "det_count"].mean():.2f}\n')
fired = (pattern.loc[X.index, [f'det_{c}' for c in config.CLASSES]] > 0).mean()
fired.index = config.CLASSES
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(fired.index, fired.values, color='#3949AB')
ax.set_ylabel('share of days'); ax.set_title('detection rate on the latest candle')
plt.xticks(rotation=35, ha='right'); plt.tight_layout(); plt.show()

## Univariate check, before any model

For each class: the next-day up-rate on days the detector fired, against the overall rate.
Wilson intervals, because some classes fire on only a handful of days and a bare percentage
would be badly misleading.

In [ ]:
def wilson(k, n, z=1.96):
    """Wilson score interval; behaves sensibly for tiny n, unlike the normal approx."""
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z**2/n
    c = (p + z**2/(2*n)) / d
    hw = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / d
    return c - hw, c + hw

rows = []
overall = y.mean()
for c in config.CLASSES:
    mask = pattern.loc[X.index, f'det_{c}'] > 0
    n = int(mask.sum())
    if n == 0:
        rows.append({'pattern': c, 'days': 0}); continue
    k = int(y[mask].sum()); lo, hi = wilson(k, n)
    rows.append({'pattern': c, 'days': n, 'up_rate': k/n,
                 'ci_low': lo, 'ci_high': hi,
                 'vs_overall': k/n - overall,
                 'distinguishable': not (lo <= overall <= hi)})
uni = pd.DataFrame(rows)
print(f'overall up-rate: {overall:.4f}')
uni

## The walk-forward comparison

Expanding window, refit every January, never shuffled. Both variants are scored on exactly
the same days so the comparison is paired.

In [ ]:
from src.downstream_signal.backtest import run_comparison

report = run_comparison(X, y, fwd,
                        out_path=config.RESULTS_DIR / 'downstream_signal_comparison.json')
b, p = report['variants']['baseline'], report['variants']['with_patterns']
pd.DataFrame({'baseline': b, 'with_patterns': p}).loc[
    ['accuracy', 'base_rate', 'roc_auc', 'f1', 'brier',
     'strategy_return_net', 'buy_hold_return', 'sharpe_net', 'n_trades']]

## Is the difference real, or is it noise?

Two accuracies differing by a few tenths of a percent over ~1,700 days are not
distinguishable by eye. McNemar's test uses only the days where the two models disagreed;
the paired bootstrap gives an interval for the difference itself.

In [ ]:
bs, mc = report['comparison']['bootstrap'], report['comparison']['mcnemar']
print(f"accuracy difference  {bs['delta_accuracy']:+.4f}")
print(f"95% bootstrap CI     {bs['ci95_low']:+.4f} to {bs['ci95_high']:+.4f}")
print(f"P(difference > 0)    {bs['prob_delta_positive']:.3f}")
print(f"McNemar p-value      {mc['p_value']:.4f}  "
      f"({mc['discordant_pairs']} discordant days)")

crosses_zero = bs['ci95_low'] <= 0 <= bs['ci95_high']
print('\nVERDICT:', 'NO measurable improvement — the interval contains zero.'
      if crosses_zero or mc['p_value'] >= 0.05 else
      'A statistically detectable difference. Judge the effect SIZE, not just the p-value.')

## Write the results into the README

In [ ]:
from src.evaluation.report import inject, build_report
print(build_report())
inject()